# The Token Budget [Step 01.02]

> **MLCourse - Agentic AI - Agent Patterns**

Notebook 01 showed *what* is in the window. This one turns that into an
engineering artifact: a **budget** with a hard ceiling, named allocations per
component, and code that enforces it.

The mental shift:

```
  WITHOUT a budget                    WITH a budget
  ----------------                    -------------
  build the prompt                    decide the ceiling      (e.g. 2000 tokens)
  send it                             allocate per component  (system 200, docs 700, ...)
  hope it fits                        fill each allocation
  discover the limit in production    enforce BEFORE sending
```

### What you'll learn

- How to write a budget as data, not as a vague intention.
- How to measure the true cost of tool schemas (with a real before/after call).
- Why a budget needs **headroom for the output**, not just the input.
- What to do when a component overflows its allocation.

### Why it matters

A budget converts an unbounded, unpredictable failure ("sometimes the request is
rejected", "sometimes it costs 4x") into a bounded, predictable one ("we dropped
the oldest three turns"). Bounded failures are the ones you can ship.

### Prerequisites

- [01_what_goes_in_the_window](01_what_goes_in_the_window.ipynb)

### Setup: environment, model, token counting, rate-limit-aware call helper


In [ ]:
import os                              # environment variables
import time                            # timing and pacing
import json                            # pretty-printing structured context
from pathlib import Path               # locating the track root
from dotenv import load_dotenv         # reads KEY=value pairs from .env

# Walk UP from the notebook folder until we hit the repo root, then load the
# (gitignored) .env that lives inside 03_agentic_ai. Note the extra path
# segment: the walk-up lands on the REPO ROOT, not on the track folder.
TRACK = Path.cwd()
while not (TRACK / "03_agentic_ai").exists() and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / "03_agentic_ai" / ".env")

GROQ_MODEL = "qwen/qwen3.8-27b"        # hosted, fast, generous free tier
# Local alternative (documented, not used here): Ollama `llama3.1:8b` via
# `from langchain_ollama import ChatOllama`. OpenAI is never used in this course.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 300, **kw):
    """One place that constructs the chat model, so every notebook is identical."""
    return ChatGroq(model=GROQ_MODEL, temperature=temperature,
                    max_tokens=max_tokens, **kw)


# --- Token counting -----------------------------------------------------------
# Two different numbers, and it matters which one you are looking at:
#   * approx_tokens(): a LOCAL estimate using tiktoken's cl100k_base. It is not
#     the model's own tokenizer, so treat it as "within ~10%", good for
#     budgeting BEFORE you send a request.
#   * usage_metadata on the response: the provider's EXACT count. Ground truth,
#     but only available AFTER you have already paid for the call.
import tiktoken

_ENC = tiktoken.get_encoding("cl100k_base")


def approx_tokens(text) -> int:
    """Approximate token count for a string (or anything str()-able)."""
    return len(_ENC.encode(str(text)))


# --- Rate-limit-aware calling --------------------------------------------------
# The Groq free tier allows 8000 tokens per minute. Several notebooks here make
# many small calls in a loop, so we self-pace well under the ceiling and retry
# with exponential backoff if we are throttled anyway.

TPM_BUDGET = 3500                       # deliberately conservative
_WINDOW = []                            # [(timestamp, tokens), ...]
USAGE = {"calls": 0, "in": 0, "out": 0, "seconds": 0.0}


def _pace(cost: int):
    """Sleep just enough that our rolling 60s token usage stays under budget."""
    now = time.time()
    while True:
        recent = [(t, n) for (t, n) in _WINDOW if now - t < 60]
        _WINDOW[:] = recent
        if sum(n for _, n in recent) + cost <= TPM_BUDGET or not recent:
            return
        time.sleep(min(5.0, 60 - (now - recent[0][0]) + 0.5))
        now = time.time()


def chat(messages, llm=None, temperature=0.0, max_tokens=300, retries=5):
    """Send `messages`, return the AIMessage. Paces, retries, and meters usage.

    `messages` is a list of (role, content) tuples or LangChain message objects.
    """
    llm = llm or make_llm(temperature=temperature, max_tokens=max_tokens)
    est = approx_tokens(messages) + max_tokens
    delay = 4.0
    for attempt in range(retries):
        _pace(est)
        t0 = time.time()
        try:
            out = llm.invoke(messages)
        except Exception as exc:
            if "rate_limit" in str(exc) or "429" in str(exc):
                time.sleep(delay)
                delay = min(delay * 2, 45)
                continue
            raise
        u = out.usage_metadata or {}
        _WINDOW.append((time.time(), u.get("total_tokens", est)))
        USAGE["calls"] += 1
        USAGE["in"] += u.get("input_tokens", 0)
        USAGE["out"] += u.get("output_tokens", 0)
        USAGE["seconds"] += time.time() - t0
        return out
    raise RuntimeError("still rate limited after %d attempts" % retries)


def ask(prompt: str, system: str = None, **kw) -> str:
    """Convenience wrapper: one user turn in, plain text out."""
    msgs = ([("system", system)] if system else []) + [("user", prompt)]
    return chat(msgs, **kw).content.strip()


print("model:", GROQ_MODEL)
print("key loaded:", bool(os.getenv("GROQ_API_KEY")))
print("tokenizer:", "cl100k_base (approximation)")


### 1. A budget is a dict

There is no framework needed. A budget is a ceiling and a set of allocations
that sum to it.

The important discipline is that the allocations are **decided in advance**, from
what each component is worth, not derived from whatever the component happens to
contain today.

In [2]:
CONTEXT_LIMIT = 2000        # total tokens we allow ourselves per request

BUDGET = {
    "system":   200,        # rules, role, format - small and stable
    "tools":    350,        # schemas for the tools this agent may call
    "docs":     600,        # retrieved evidence
    "history":  500,        # prior conversation turns
    "query":    100,        # the user's new message
    "output":   250,        # RESERVED for the model's reply
}

assert sum(BUDGET.values()) == CONTEXT_LIMIT, sum(BUDGET.values())

for k, v in BUDGET.items():
    print("%-9s %5d  %s" % (k, v, "#" * (v // 25)))
print("%-9s %5d" % ("TOTAL", sum(BUDGET.values())))

system      200  ########
tools       350  ##############
docs        600  ########################
history     500  ####################
query       100  ####
output      250  ##########
TOTAL      2000


> **The output allocation is the one everyone forgets.** The context limit covers
> input *plus* generated output. If you fill the window to the brim with input,
> the model has nowhere to write the answer and you get a truncated response - or
> a hard error - that looks nothing like a budgeting bug.

### 2. Measuring the real cost of tool schemas

In notebook 01 we estimated tool cost locally. Now let us get the provider's own
number, by sending the *same* question twice - once with tools bound, once
without - and reading `input_tokens` off each response.

This is the honest way to price a feature: A/B the request and read the meter.

In [3]:
from langchain_core.tools import tool


@tool
def lookup_order(order_id: str) -> str:
    """Fetch the status, items and delivery estimate for one order.

    Args:
        order_id: Order id, e.g. NW-10231.
    """
    return "shipped, arriving Thursday"


@tool
def start_return(order_id: str, sku: str, reason: str) -> str:
    """Open a return request for one item on an order.

    Args:
        order_id: Order id.
        sku: Product SKU to return.
        reason: The customer's stated reason for returning it.
    """
    return "return RMA-88 opened"


@tool
def check_stock(sku: str, warehouse: str = "any") -> str:
    """Report how many units of a product are in stock in a warehouse.

    Args:
        sku: Product SKU.
        warehouse: Warehouse code, or "any".
    """
    return "12 units"


QUESTION = "What is the status of order NW-10231?"

bare = chat([("user", QUESTION)], max_tokens=60)
with_tools = chat([("user", QUESTION)],
                  llm=make_llm(max_tokens=60).bind_tools(
                      [lookup_order, start_return, check_stock]),
                  max_tokens=60)

a = bare.usage_metadata["input_tokens"]
b = with_tools.usage_metadata["input_tokens"]
print("input tokens, no tools   :", a)
print("input tokens, 3 tools    :", b)
print("cost of the tool schemas : %d tokens (%.1fx the bare prompt)" % (b - a, b / a))
print("per tool, roughly        : %d tokens" % ((b - a) / 3))

input tokens, no tools   : 26
input tokens, 3 tools    : 540
cost of the tool schemas : 514 tokens (20.8x the bare prompt)
per tool, roughly        : 171 tokens


Now project that. Tool schemas are resent on **every** turn of every session.

In [4]:
per_tool = (b - a) / 3
for n_tools in (3, 10, 25, 50):
    per_call = per_tool * n_tools
    print("%3d tools -> %6.0f tokens/call, %7.0f tokens over a 20-turn session"
          % (n_tools, per_call, per_call * 20))
print()
print("Budget allocation for tools was %d tokens, which buys about %d tools."
      % (BUDGET["tools"], BUDGET["tools"] / per_tool))

  3 tools ->    514 tokens/call,   10280 tokens over a 20-turn session
 10 tools ->   1713 tokens/call,   34267 tokens over a 20-turn session
 25 tools ->   4283 tokens/call,   85667 tokens over a 20-turn session
 50 tools ->   8567 tokens/call,  171333 tokens over a 20-turn session

Budget allocation for tools was 350 tokens, which buys about 2 tools.


This is the measured argument for **semantic routing** and tool subsetting: you
do not bind all 50 tools, you bind the 5 that could plausibly be relevant to this
request. The budget makes the trade-off arithmetic instead of opinion.

### 3. Enforcing the budget

A budget that is not checked is a comment. Here is a small enforcer: it measures
each component, reports which ones overflow, and refuses to build an
over-budget request.

Note the design choice - it **reports every** violation rather than raising on
the first one. When you are debugging a bloated agent you want the whole picture.

In [5]:
def audit(components: dict, budget: dict):
    """Compare each component against its allocation. Returns (ok, report rows)."""
    rows, ok = [], True
    for name, allocation in budget.items():
        if name == "output":
            rows.append((name, 0, allocation, "reserved"))
            continue
        used = approx_tokens(components.get(name, ""))
        status = "ok" if used <= allocation else "OVER by %d" % (used - allocation)
        if used > allocation:
            ok = False
        rows.append((name, used, allocation, status))
    return ok, rows


def show(components, budget):
    ok, rows = audit(components, budget)
    print("%-9s %8s %8s   %s" % ("component", "used", "budget", "status"))
    print("-" * 48)
    for name, used, alloc, status in rows:
        print("%-9s %8d %8d   %s" % (name, used, alloc, status))
    total = sum(r[1] for r in rows) + budget["output"]
    print("-" * 48)
    print("%-9s %8d %8d   %s" % ("TOTAL", total, CONTEXT_LIMIT,
                                 "ok" if total <= CONTEXT_LIMIT else "OVER"))
    print("\nwithin budget:", ok)
    return ok


SYSTEM = ("You are Ada, the support assistant for Northwind Cycles. Answer only "
          "from the reference material. If it is not there, say you don't know. "
          "At most three sentences. End with one follow-up question.")

# Eight retrieved policy paragraphs. A real retriever returning top-8 produces
# exactly this much text, and it does not fit our 600-token allocation.
DOCS = [
    "Returns policy: unused items may be returned within 30 days of delivery, in "
    "original packaging. Refunds reach the original payment method within 5 "
    "business days of arrival at our warehouse. Return shipping is free within the EU.",
    "Shipping: standard delivery is 3-5 business days within the EU. Express is "
    "1-2 business days for an extra 9 EUR. Orders placed after 15:00 CET ship the "
    "next business day. We do not ship to PO boxes or to addresses outside the EU.",
    "Warranty: frames carry a 5-year warranty against manufacturing defects. Wheels "
    "and drivetrain components carry 2 years. Wear items such as tyres, brake pads, "
    "chains and cables are not covered. Warranty claims require the original invoice.",
    "Assembly: bikes ship 85% assembled. The customer fits the front wheel, "
    "handlebars, pedals and saddle. A torque wrench is required for carbon parts. "
    "Free assembly is available at any partner workshop within 60 days of delivery.",
    "Payment: we accept card, SEPA transfer and instalments over 3, 6 or 12 months. "
    "Instalment plans require a credit check and are unavailable on orders under "
    "300 EUR. Refunds on instalment orders cancel the remaining schedule.",
    "Price matching: we match any verified EU retailer price on identical stock "
    "items within 14 days of purchase. Clearance, ex-demo and auction listings are "
    "excluded. The customer must supply a link showing the item in stock.",
    "Cancellation: orders can be cancelled free of charge until they enter picking, "
    "usually within 2 hours of ordering. After dispatch, a cancellation becomes a "
    "return and follows the returns policy above.",
    "Damage in transit: report visible damage within 48 hours of delivery with "
    "photographs of the packaging and the item. We arrange collection and send a "
    "replacement without waiting for the damaged unit to reach the warehouse.",
]

HISTORY_TEXT = "\n".join(
    "%s: %s" % (r, t) for r, t in
    [("USER", "Hi, I want to check on my order NW-10231."),
     ("ASSISTANT", "Of course. It shipped yesterday and is due Thursday."),
     ("USER", "Great. Was the express option applied?"),
     ("ASSISTANT", "No, it went out on standard delivery, 3-5 business days."),
     ("USER", "I thought I paid for express. Can you check?"),
     ("ASSISTANT", "The order shows standard shipping and no express surcharge."),
     ("USER", "Annoying. Can I upgrade it now?"),
     ("ASSISTANT", "Not once it has been dispatched, unfortunately."),
     ("USER", "Fine. Is the bike assembled?"),
     ("ASSISTANT", "It arrives 85% assembled; you fit the wheel, bars, pedals and saddle."),
     ("USER", "Do I need special tools?"),
     ("ASSISTANT", "A torque wrench for the carbon parts, otherwise basic allen keys."),
     ("USER", "And if something is damaged in the box?"),
     ("ASSISTANT", "Photograph it within 48 hours and we send a replacement straight away.")])

current = {
    "system": SYSTEM,
    "tools": json.dumps([lookup_order.args_schema.model_json_schema(),
                         start_return.args_schema.model_json_schema(),
                         check_stock.args_schema.model_json_schema()]),
    "docs": "\n".join(DOCS),
    "history": HISTORY_TEXT,
    "query": "How long do I have to return the bike?",
}

_ = show(current, BUDGET)

component     used   budget   status
------------------------------------------------
system          44      200   ok
tools          307      350   ok
docs           369      600   ok
history        199      500   ok
query           10      100   ok
output           0      250   reserved
------------------------------------------------
TOTAL         1179     2000   ok

within budget: True


### 4. What to do when a component overflows

There are exactly four moves, and they are not interchangeable:

| Move | What it means | Use when |
|---|---|---|
| **Drop** | Remove whole items | History turns, low-scoring docs |
| **Compress** | Replace items with a summary | History (see module 02) |
| **Reallocate** | Take budget from another component | One component is chronically starved |
| **Raise the ceiling** | Use a bigger window / model | Genuinely need it, and can pay |

The mistake is reaching for "raise the ceiling" first. It is the most expensive
move and, as notebook 04 shows, it does not even reliably work.

Below we apply *drop* to the two offenders and re-audit.

In [6]:
def fit(text: str, allocation: int) -> str:
    """Truncate text to fit an allocation, on a token boundary."""
    ids = _ENC.encode(text)
    if len(ids) <= allocation:
        return text
    return _ENC.decode(ids[:allocation])


def drop_oldest_turns(history_text: str, allocation: int) -> str:
    """Drop whole turns from the FRONT until the history fits."""
    lines = history_text.split("\n")
    while lines and approx_tokens("\n".join(lines)) > allocation:
        lines.pop(0)
    return "\n".join(lines)


def drop_docs(docs: list, allocation: int) -> str:
    """Keep whole documents, best-first, until the next one would not fit."""
    kept = []
    for d in docs:
        if approx_tokens("\n".join(kept + [d])) > allocation:
            break
        kept.append(d)
    return "\n".join(kept)


fixed = dict(current)
fixed["docs"] = drop_docs(DOCS, BUDGET["docs"])
fixed["history"] = drop_oldest_turns(HISTORY_TEXT, BUDGET["history"])

print("docs: kept %d of %d documents" % (fixed["docs"].count("Returns policy") , len(DOCS)))
print("history: kept %d of %d lines\n" % (len(fixed["history"].split("\n")),
                                          len(HISTORY_TEXT.split("\n"))))
show(fixed, BUDGET)

docs: kept 1 of 8 documents
history: kept 14 of 14 lines

component     used   budget   status
------------------------------------------------
system          44      200   ok
tools          307      350   ok
docs           369      600   ok
history        199      500   ok
query           10      100   ok
output           0      250   reserved
------------------------------------------------
TOTAL         1179     2000   ok

within budget: True


True

> **Pitfall: truncating mid-item.** `fit()` above cuts on a token boundary, which
> is safe for *characters* but not for *meaning* - you can end up with half a
> policy sentence that reverses its own meaning ("unused items may be returned
> within 30 days of delivery, in original packaging. Refunds reach the ori").
> Prefer dropping whole units (a turn, a document) over slicing one in half.
> `drop_docs` and `drop_oldest_turns` do that; `fit` is the last resort.

### 5. Does the trimmed context still answer the question?

A budget that produces a wrong answer is not a win. So we check: send the
budgeted context and see if the answer is still correct and grounded.

In [7]:
prompt = (fixed["system"]
          + "\n\nReference material:\n" + fixed["docs"]
          + "\n\nConversation so far:\n" + fixed["history"])

reply = chat([("system", prompt), ("user", fixed["query"])], max_tokens=BUDGET["output"])

print("input tokens billed :", reply.usage_metadata["input_tokens"])
print("output tokens billed:", reply.usage_metadata["output_tokens"])
print("ceiling             :", CONTEXT_LIMIT)
print()
print(reply.content)
print()
print("grounded in the kept doc ('30 days'):", "30" in reply.content)

input tokens billed : 683
output tokens billed: 43
ceiling             : 2000

You have 30 days from delivery to return the bike, provided it is unused and in its original packaging. Return shipping is free within the EU. Would you like to know how the refund is processed?

grounded in the kept doc ('30 days'): True


### 6. Pitfalls

- **No output reservation.** Input + output share the window. Reserve for output.
- **Budgeting the average.** Budget the *worst realistic* case; averages do not
  get rejected by the API, tails do.
- **Enforcing after assembly only.** Check before you send, every time - the
  check costs microseconds and the call costs money.
- **Chronic starvation.** If one component overflows on 90% of requests, that is
  not a trimming problem, that is a mis-set allocation. Reallocate.

### Recap

| Idea | Takeaway |
|---|---|
| Budget as data | A ceiling plus named allocations that sum to it |
| Reserve output | The window covers generation too |
| Tools are a tax | Measured here with an A/B call; ~`per_tool` tokens each, per turn |
| Four moves | Drop, compress, reallocate, raise the ceiling - in that order |
| Drop whole units | Never slice a document or a turn in half |

**Next:** [03_trimming_strategies](03_trimming_strategies.ipynb) - three different
answers to "which items do I drop?", compared on the same over-budget context.